In [1]:
from src.cv_model import run_ablation_experiment
from src.utils.data_processing import load_data_base,add_features
import pandas as pd
from config import DATA_PATH

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/tiphainell/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
# Notebook d'ablation study

# Objectif :
# Comparer différentes configurations de modèles afin d'évaluer l'apport des features.

# Études réalisées :
# - Baseline : UltimateClaim = InitialClaim
# - Modèle avec uniquement les features démographiques
# - Modèle avec features démographiques + NLP
# - Modèle avec uniquement les features NLP
# - Modèle démographique + NLP sans la variable InitialClaim (feature très informative)

# Objectif principal :
# Mesurer l'impact de chaque groupe de variables sur la performance du modèle en cross-validation (5 folds).

In [3]:
# loading de toute la base
database = load_data_base(DATA_PATH)

# features engineering, ajout des features nlp si nlp=True
database, list_features_demog_x, list_features_nlp = add_features(database, True)

database=database[database['role']=="train"]

In [4]:
#définition des features que NLP
list_nlp=[x for x in list_features_nlp if x not in list_features_demog_x]
features_without_baseline = [
    x for x in list_features_nlp
    if x != "InitialIncurredCalimsCost"
]


In [5]:
#grid, best_model=run_xgb_gridsearch(database[features_nlp_x],database["UltimateIncurredClaimCost"])

In [6]:
#Défintion des sous catégories de features
input_subcategories={"baseline": "InitialIncurredCalimsCost",
                     "demog_only":list_features_demog_x,
                     "demog + NLP":list_features_nlp,
                     "NLP only": list_nlp,
                     "without baseline": features_without_baseline
                     }

In [7]:
#Lancement de la cross-validation sur toutes les features
metrics, model= run_ablation_experiment(5,input_subcategories,"UltimateIncurredClaimCost",database)


--- Testing category: baseline ---

--- Testing category: demog_only ---

--- Testing category: demog + NLP ---

--- Testing category: NLP only ---

--- Testing category: without baseline ---


In [8]:
#RMSE sur les  5 folds

rmse_table = pd.DataFrame({
    category: metrics[category]["RMSE"]
    for category in metrics
})

print(rmse_table.round(0))


   baseline  demog_only  demog + NLP  NLP only  without baseline
0   27869.0     25944.0      25670.0   29972.0           25559.0
1   26503.0     25758.0      25566.0   29760.0           25856.0
2   45010.0     44442.0      44354.0   47048.0           44370.0
3   25701.0     23898.0      23659.0   28152.0           23881.0
4   29375.0     22428.0      22146.0   27491.0           22540.0


In [9]:
#Synthèse RMSE sur les 5 folds
rmse_mean = rmse_table.mean()
rmse_std = rmse_table.std()

summary_table = (
    rmse_table
    .agg(["mean", "std"])
    .T
    .round(0)
    .astype(int)
)

print(summary_table)

                   mean   std
baseline          30891  8015
demog_only        28494  9031
demog + NLP       28279  9104
NLP only          32485  8209
without baseline  28441  9005
